# 05.5 — Tests, evaluation, and review

[Class index and setup](README.md) · [Previous: 05.4](05.4-guided-failure-practice.ipynb)

**Purpose:** Leave honest test and evaluation evidence for the small application.

**In class:** 0 minutes. **Independent work:** 75 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's project is additional.

**Starting knowledge:** Basic Python functions, dictionaries, JSON/schema validation, and the earlier pytest ideas. All code and saved inputs are supplied again; no earlier exercise answers or paid key are needed.

**Learning outcomes:** Run deterministic checks; compare five expected/observed outcomes; assign one failure to the right layer.

[Glossary](GLOSSARY.md) · [Supplied code](report_app/__init__.py) · [Fixture provenance](fixtures/README.md)

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/harness.py").is_file() or not Path("fixtures/world-bank-population.json").is_file():
    raise FileNotFoundError(
        "Open 05-small-ai-applications.code-workspace and restart the kernel. "
        "Download the complete class folder with report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available. Default execution is REPLAY, with no HTTP or credentials.")

## Independent work · 75 minutes

Exercise 05.5.H1 includes **15 minutes** running/reading the supplied tests, then **45 minutes** evaluating five prompts. Exercise 05.5.H2 uses **15 minutes** for reflection and the completion check. These finish the 60-minute test/failure allocation, the 45-minute evaluation allocation, and the 15-minute reflection allocation. No extra setup or homework is added.

### Supplied tests — 15 minutes

Run the suite below. It blocks real HTTP and checks the actual allowlist, argument/final-answer validation, counters, and replay/tool-result agreement. The SDK transport tests exercise the installed SDK's JSON codec using a synthetic HTTP transport, including error behavior and retry settings. They establish application/SDK integration, **not** successful access to a real model.

Spend five minutes running the suite, five reading the third-tool and unknown-name checks, and five identifying which boundary is controlled by the test. Students only complete the simple assertion from part 05.3; writing a new test suite is not required.

In [ ]:
import subprocess
import sys

completed = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True, timeout=60)
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
assert completed.returncode == 0, "Inspect the supplied tests' actual failure before continuing."

### Evaluate five prompts — 45 minutes

Use nine minutes per row: predict the expected operation and result, inspect the observed answer and trace, decide whether the expectation is met, and record a short reason. This includes the table-writing time. Work on the five prompts already supplied:

| Case | Expected behavior |
| --- | --- |
| Portugal | Find PRT:2023; report 10,578,174 persons and the saved year |
| Total | Use summary; report 58,930,702 persons for these two countries |
| Comparison | Use two exact lookups; Spain is larger; report both values |
| Missing France | Explain absent saved data; do not invent a population or use zero |
| Modify | Explain read-only scope; execute no tool; change no data |

Expectations constrain behavior and facts, not an exact sentence. A different live wording can meet the expectation. A factually wrong string can pass the schema, so inspect both the answer and the actual tool results. No automated string match or LLM judge grades your prose here.

The table produced below contains saved scripts. Label it **illustrative replay**, and explain that the model's choice and wording were authored in advance. Changing the prompt cannot evaluate a new model response through a fixed script. If live access is available, use the explicit command in [AI_SETUP.md](AI_SETUP.md) and create a separately labelled table with model ID and date. Required work does not depend on that access.

In [ ]:
from report_app.evaluation import evaluate

rows = evaluate()
for row in rows:
    result = row["result"]
    print("CASE:", row["case"], "MODE:", result["mode"])
    print("EXPECTED:", row["expected"])
    print("OBSERVED:", row["observed"])
    print("REQUESTS/TOOLS:", result["model_requests"], result["tool_executions"])
    print("TRACE:", result["trace"])

Record this table as comments in your answer cell or in a scratch Markdown file:

| Case | Expected behavior | Observed answer / tools | Evidence mode | Met? Why? |
| --- | --- | --- | --- | --- |
| Portugal | Correct saved value/year | Fill from trace | Illustrative replay | Your judgment |
| Total | Correct two-country total | Fill from trace | Illustrative replay | Your judgment |
| Comparison | Both values; Spain larger | Fill from trace | Illustrative replay | Your judgment |
| Missing | Absent, not zero | Fill from trace | Illustrative replay | Your judgment |
| Modify | Read-only; zero tools | Fill from trace | Illustrative replay | Your judgment |

### Exercise 05.5.H1 — Record expected versus observed

> **Independent work · 60 minutes: 15 tests + 45 evaluation**
>
> **Goal:** Leave a small evidence table without adding application code.
>
> **Start here:** The supplied test run, five rows above, and table template above.
>
> **Task:** For each row, record whether the expected behavior was met and why. Check numbers and tool executions against the trace. Keep every row labelled replay; if you opt into live access, record its evidence in a separate table.
>
> **Check:** Five rows include expected behavior, observed result, replay/live evidence, and your reasoned judgment.
>
> **Hint:** No new function is needed. The comparison uses two lookups; the modification answer should execute zero tools.

In [ ]:
# Your attempt for 05.5.H1.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_5_h1.py

### Visual guide — Match the claim to its evidence

<img src="images/evidence-types.png" alt="Replay supports scenario and policy checks; mocked SDK transport supports codec integration; live evaluation supports dated observations of a selected model." width="552" style="max-width: 100%; height: auto; background-color: white;">

Replay exercises the real application with authored responses. Mocked SDK transport checks request/response integration. Only a live evaluation observes the selected model's behavior on that date; none of these alone proves broad answer quality.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/evidence-types.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Assign the fix to the responsible layer

A **schema** fix changes which shapes/types are accepted. A **tool** fix changes a deterministic operation or its data handling. A **prompt** fix changes instructions to the model, which must be checked through live behavior before claiming improvement. An **application-logic** fix changes dispatch, counters, or error handling.

For example, a numeric `answer` is an output-contract failure. A tool adding the wrong observations is a deterministic calculation failure. A schema-valid sentence inventing a population is a factual-behavior failure; compare it with the tool data, inspect context/instructions, and evaluate any prompt change live. A third executed tool is an application limit failure. Do not assign all failures to prompts.

### Exercise 05.5.H2 — Explain one failure

> **Independent work · 15 minutes including completion review**
>
> **Goal:** Connect a concrete failure to its responsible layer.
>
> **Start here:** One supplied failed trace from 05.4, plus your evaluation table.
>
> **Task:** Spend 5 minutes selecting the trace and quoting the failed expectation, 5 assigning the fix to schema/tool/prompt/application logic, and 5 completing the checklist below. Write a short explanation; no new code is required.
>
> **Check:** Explain the failure, layer, proposed fix, and what test/evaluation would provide evidence for the fix.
>
> **Hint:** For a numeric lookup label, inspect the argument schema; for an extra executed tool, inspect the loop guard.

In [ ]:
# Your attempt for 05.5.H2.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_5_h2.py

## Required completion check — inside the final 15 minutes

- Show the one-field Answer and explain why shape does not establish truth.
- Show the two permitted read-only tools and one rejected input.
- Identify the two-execution/three-request limits in an inspected trace.
- Run the supplied deterministic checks and retain their actual result.
- Supply five expected/observed rows with honest replay/live labels.
- Explain one failure and how you would check a proposed fix.

A peer should be able to run this complete folder without earlier answers, a paid key, a model download, or a GPU. AI coding assistants may help; explain and modify the tiny completed lines yourself. Larger tool/loop implementations are optional and do not change required completion.

## Optional extension — separate from required work

Use the index's combined 180-minute ceiling: compare one prompt revision live (up to 45), explore CPU Ollama with the separate guide (up to 120 including downloads), or inspect one additional saved scenario (up to 15). No specific local model is recommended without CPU tool/schema verification. Keep live/model delivery evidence separate from deterministic tests.

[Instructor notes](INSTRUCTOR.md) · [Optional/live setup](AI_SETUP.md) · [Fixture provenance](fixtures/README.md)